# High-performance and parallel computing for AI - JAX profiling tutorial

IMPORTANT
=========

* When using a GPU we need to set some environment variables (see below). If you get some weird error restart the kernel and rerun the code below.
* For these practicals and tutorials we will be using a different `conda environment`. When opening a notebook or a terminal make sure you are using the **hpc4ai Kernel**!!!

Before you start (and before running any other GPU code on the servers) please run the following code, which attempts to limit the maximum GPU memory usage and picks two L40s GPUs at random, excluding the Quadro GPUs. **Please only run the code below once every time you restart the kernel!**

**NOTE:** Here we are setting the `JAX_NUM_CPU_DEVICES` environment variable to tell JAX we want it to see more than one CPU.

In [1]:
import os

# JAX-specific environment variables
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.249" # restrict GPU memory preallocation to a fourth of the total
os.environ["JAX_OPTIMIZATION_LEVEL"]="O1"
os.environ["JAX_NUM_CPU_DEVICES"] = "6" # JAX will see 6 CPUs (if available)

## On goliat we have FIVE GPUs so here we pick two of those at random
## so that we do not overload the system.
## The way we do it is by figuring out the GPU UUIDs and then setting
## The CUDA_VISIBLE_DEVICES environment variable.
## Note: this is useful for other libraries as well (e.g., Jax, PyTorch, TF) in multi-GPU servers.

# To get these UUIDs you need to run nvidia-smi -q on the command line
quadro_UUIDs = ["GPU-4efa947b-abbd-7c6e-84f5-61241d34bb4b",
                "GPU-5eb524b0-2b1b-fe98-e6ed-b8fb5185e993"]

L40s_UUIDs = ["GPU-7bba1f33-03d2-016b-d42e-ced83c3ac243",
              "GPU-179d068a-3bea-91d7-1a8c-7017f55d6298",
              "GPU-ae634859-dd49-de46-9182-195639405eaa"]

import random

a, b = random.sample(range(3), 2) # picks two numbers between 0 and 2 at random

# Picks an L40s and a Quadro GPU at random. The others will be invisible to JAX
# NOTE: this only works if the environment variable is set BEFORE JAX is first imported.
os.environ["CUDA_VISIBLE_DEVICES"] = L40s_UUIDs[a] + "," + L40s_UUIDs[b]

## JAX (or whatever other software) will only see these GPUs.

############################ IMPORTS #############################

import jax
import jax.numpy as jnp
import shutil # used for file management to circumvent some jupyterhub remote server issues - not usually needed

gpus = jax.devices("gpu")
cpus = jax.devices("cpu")

assert len(cpus) == 6 and len(gpus) == 2

# JAX profiling

A *profiler* is a software tool that allows a programmer to inspect how much time and memory are spent for executing each portion of a code.

JAX can profile both performance and memory at the same time as well as all device computations. This is unusual and a great feature!

To enable JAX profiling, use

```python
with jax.profiler.trace(foldername):
    # MY CODE
```

or, alternatively

```python
jax.profiler.start_trace(foldername)
# MY CODE
jax.profiler.stop_trace()
```

To enable generation of [perfetto](https://ui.perfetto.dev/)-compatible tracing data, invoke the profiler with the flag
`create_perfetto_trace=True`. For instance:

```python
with jax.profiler.trace(foldername, create_perfetto_trace=True):
    # MY CODE
```

For use within this Jupyter server we will also need the flag `create_perfetto_link=False` since otherwise JAX will spawn a link to perfetto and will
stop execution until the webpage is closed. Since we are working in an isolated environment we cannot access that webpage so it would hang. To avoid this, we have to pass this additional flag.

## Analysing JAX traces in JupyterHub

The existing JAX-compatible profilers, `xperf` and `perfetto`, both work by spawning a webpage containing profiler results. This webpage is hosted locally where the profiler command is run and must be accessed from a browser.

If we did this on our laptop we would be fine, but since we are in a protected environment (the JupyterHub we are using for the practicals), we need one of the following two workarounds.

### Option 1: Download tracing data and run the profiler on your laptop

Create a zip file with the JAX-generated tracing data, download it, and run the profiler on your laptop.

This requires a local installation of `xprof`/`tensorboard` or using `perfetto` (which is worse, but does not require any installation).

To install tensorboard with `xprof` support on Linux:

```bash
python3 -m pip install tensorboard tensorboard-plugin-profile
```

You can then open a terminal in your laptop and run

```bash
tensorboard --logdir=./jax_trace --port=6007 # can replace tensorboard with xprof, it's the same here.
```

Here `./jax_trace` is the folder where the trace was saved and `port` is the port used to spawn the webserver.

### Option 2: Use a proxy server and call the profiler from the jupyterhub terminal (tensorboard only).

Open a terminal in JupyterHub (click on the plus sign), and activate the hpc4ai conda environment:

```bash
conda activate hpc4ai
```

Then run tensorboard from terminal:

```bash
tensorboard --logdir=./jax_trace --port=6007 # CANNOT replace tensorboard with xprof for this option.
```

Load the profiler by opening the following link in your browser:

[http://10.10.15.20:8000/hub/user-redirect/proxy/6007/](http://10.10.15.20:8000/hub/user-redirect/proxy/6007/)

**NOTE:** The slash at the end is **crucial**.

<details>
<summary>
Notes for Matteo with instructions to follow in order for Option 2 to work for everyone
</summary>

* requires `jupyter-server-proxy` extension to be installed and activated in the JupyterHub environment
* requires installing tensorboard with xprof support: `sudo -E python3 -m pip install tensorboard tensorboard-plugin-profile`
</details>

## Example 1 - Matmul profiling example

Here is an example of profiling a simple function in JAX: a matmul followed by a tanh activation function.

**Important:** The use of `jax.block_until_ready()` is crucial here since JAX works asyncronously!

**Note:** In order to also support Option 1 above we use the Python built-in `shutil` module for file operations.

In [2]:
# where we save the trace
jax_trace_folder = "jax_trace"
try: shutil.rmtree(jax_trace_folder) # remove previously saved traces
except FileNotFoundError: pass

key = jax.random.PRNGKey(0)
batch_size = 64
n = 2048

x = jax.random.normal(key, (batch_size, n, n), dtype=jnp.float32)
w = jax.random.normal(key, (n, n), dtype=jnp.float32)

# Batched matrix-matrix multiplication
@jax.jit
def step(x, w):
    y = jnp.matmul(x, w)
    return jnp.tanh(y)

# Warmup to trigger compilation and GPU allocation before profiling
y = step(x, w).block_until_ready()

with jax.profiler.trace(jax_trace_folder, create_perfetto_trace=True, create_perfetto_link=False): # the latter flag is only because we are running it on a server
    for i in range(10):
        y = step(x, w)
    
    y.block_until_ready()

# download, unzip, and open with xprof --logdir ./jax_trace or with https://ui.perfetto.dev/
shutil.make_archive(jax_trace_folder, "zip", root_dir=".", base_dir=jax_trace_folder) 
print("Profile trace saved!")

Profile trace saved!


### Some comments

This is a good first example, but note that the profiler does not give us much information.

The reason is that the function we profile is very simple and very fast so there is not much to say.

Next, a more involved example.

## Example 2 - A more involved profiling example: neural network training

We now take the example from a previous tutorial and profile the training of a neural network for a data fitting problem.

By changing the jax default device, we can profile execution on both CPU and GPU.

In [3]:
import optax
import equinox as eqx

# where we save the trace
jax_trace_folder = "jax_trace"
try: shutil.rmtree(jax_trace_folder) # remove previously saved traces
except FileNotFoundError: pass

def fit_function(device=None, lr=1e-3, epochs=5000):
    if device is not None:
        jax.config.update("jax_default_device", device)

    # Synthetic data: y = sin(2x) + 0.3 x^3 - 0.5 x + noise
    key = jax.random.PRNGKey(0)
    key_model, key_noise = jax.random.split(key)

    x = jnp.linspace(-2.0, 2.0, 256).reshape(-1, 1)   # fp32 by default
    y_true = jnp.sin(2.0 * x) + 0.3 * x**3 - 0.5 * x
    y = y_true + 0.05 * jax.random.normal(key_noise, y_true.shape)

    # Fully connected network: 1 -> 64 -> 64 -> 1
    model = eqx.nn.MLP(
        in_size=1,
        out_size=1,
        width_size=64,
        depth=2,
        activation=jax.nn.tanh,
        key=key_model,
    )

    optimizer = optax.adam(lr)
    opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

    def predict(model, x):
        return jax.vmap(model)(x)

    def loss_fn(model, x, y):
        pred = predict(model, x)
        return jnp.mean((pred - y) ** 2)

    @eqx.filter_jit
    def step(model, opt_state, x, y):
        loss, grads = eqx.filter_value_and_grad(loss_fn)(model, x, y)
        updates, opt_state = optimizer.update(
            grads, opt_state, eqx.filter(model, eqx.is_array)
        )
        model = eqx.apply_updates(model, updates)
        return model, opt_state, loss

    with jax.profiler.trace(jax_trace_folder, create_perfetto_trace=True, create_perfetto_link=False):
        for epoch in range(epochs):
            model, opt_state, loss = step(model, opt_state, x, y)

        jax.block_until_ready(loss)

    print(f"Final loss: {float(loss):.6e}")
    print(f"Device used for training: {jax.devices()[0] if device is None else device}")

    # download, unzip, and open with xprof -l ./jax_trace or with https://ui.perfetto.dev/
    shutil.make_archive(jax_trace_folder, "zip", root_dir=".", base_dir=jax_trace_folder) 
    print("Profile trace saved!")

    return model

if __name__ == "__main__":
    fit_function(gpus[0])

Final loss: 2.348785e-03
Device used for training: cuda:0
Profile trace saved!


### Comment

**We now get much more information!**

* Note that most of the time is obviously spent on evaluating the neural network... well, not quite, it is spent evaluating its gradient! Note the vjp and jvp in the profile trace. These come from JAX autodiff capabilities.

* If you run the code on the GPU you will see that the CPU (the host) still does some computations. Indeed the host controls the computations and sends commands to the GPU and moves memory to/from the device if needed.

* When using `xprof` we can now see more information, e.g., we can look at the roofline plot, the computation graph, and the memory profiling results.